# 02. 환경 요인 검증: 수온·기상 변수를 넣은 모델 vs 뺀 모델 (제안서 3.6절)
- 넣은 모델: 입력 변수 11개 (01과 동일)
- 뺀 모델: 환경 변수 6개(수온_주평균, 수온_평년차, 수온_변화_1주, 기온_주평균, 풍속_주평균, 유의파고_주평균)를 뺀 5개
- 두 모델 모두 하이퍼파라미터를 valid PR-AUC로 각각 다시 선택, test는 한 번만 평가
- test 차이의 불확실성: 같은 test 행을 2,000회 복원추출(부트스트랩)해 PR-AUC 차이의 95% 구간 계산
- 실행 시간 약 1분

## 공통 설정·전처리
01_모델비교와 같은 설정과 함수예요. 이 노트북만 따로 실행해도 되도록 그대로 넣었어요.

In [1]:
import itertools
import os
import warnings

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, precision_score,
                             recall_score, roc_auc_score)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

In [2]:
# ============================== [설정] ==============================
DATA_PATH = "학습데이터.csv"      # 최종 학습데이터 파일 (.csv 또는 .xlsx, 열 이름은 v_학습데이터 뷰와 같아야 함)
SHEET_NAME = 0                    # .xlsx일 때 읽을 시트 이름 또는 번호
OUT_DIR = "결과"

# 입력 변수 11개 (제안서 3.6절). 수온_변화_1주만 코드에서 만들고 나머지는 파일의 열 그대로 사용
FEATURES = [
    "통합", "통합_1주전", "통합_2주전", "통합_변화_1주",   # 현재·1~2주 전 출현율과 변화량
    "연중주차",                                          # 계절
    "수온_주평균", "수온_평년차", "수온_변화_1주",          # 수온·수온 평년차·수온 변화량
    "기온_주평균", "풍속_주평균", "유의파고_주평균",         # 기온·풍속·파고
]
BASELINE_FEATURES = ["통합"]      # 기준선: 현재 출현율만 사용 (비교용)

HORIZONS = [1, 2, 4]              # 1·2·4주 뒤 예측
TARGET = "타깃_급증_{k}주후"        # 정답 열 이름 형식
DATE_COL = "시작일"                # 시간 순서 분할 기준 열
SPLIT_RATIO = (0.7, 0.1, 0.2)     # train : valid : test
EXCLUDE_SPLIT = ["추가(2024)"]     # '분할' 열에서 이 값인 행은 제외 (2024년 5월 보고 종료로 생긴 짧은 구간)
THRESH = 0.5                      # 재현율·정밀도 계산용 확률 임계값
SEED = 42

# 데이터 오류는 DB 전처리(전처리_보정.py)에서만 고치고, 여기서는 고쳐졌는지 확인만 한다
#   포항 부이(22106) 2018-12-06~16 수온 센서 고장 → DB에서 해당 주 수온 NULL
#   → 이 기간 주평균을 쓰는 경북 2018-11-30, 2018-12-07 보고의 수온이 비어 있어야 함
SENSOR_CHECK = [{"관측소ID": 22106, "시작일": ["2018-11-30", "2018-12-07"]}]
TEMP_COLS = ["수온_주평균", "수온_평년차"]
os.makedirs(OUT_DIR, exist_ok=True)

In [3]:
def load_data(path=DATA_PATH):
    """파일을 읽고 필요한 열을 확인한 뒤, '추가(2024)' 행을 빼고 권역·날짜 순으로 정렬한다."""
    if path.lower().endswith((".xlsx", ".xls")):
        df = pd.read_excel(path, sheet_name=SHEET_NAME)
    else:
        try:
            df = pd.read_csv(path, encoding="utf-8-sig")
        except UnicodeDecodeError:
            df = pd.read_csv(path, encoding="cp949")
    df.columns = df.columns.str.strip()

    base_cols = [c for c in FEATURES if c != "수온_변화_1주"]
    need = ["권역명", DATE_COL] + base_cols + [TARGET.format(k=k) for k in HORIZONS]
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise ValueError(f"파일에 다음 열이 없습니다: {missing}")

    if "분할" in df.columns and EXCLUDE_SPLIT:
        df = df[~df["분할"].isin(EXCLUDE_SPLIT)]
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])
    df = df.sort_values(["권역명", DATE_COL]).reset_index(drop=True)
    check_sensor_errors(df)
    return df


def check_sensor_errors(df):
    """DB에서 센서 고장값이 결측 처리됐는지 확인만 한다 (값은 바꾸지 않음)."""
    for e in SENSOR_CHECK:
        m = (df["관측소ID"] == e["관측소ID"]) & df[DATE_COL].isin(pd.to_datetime(e["시작일"]))
        left = df.loc[m, TEMP_COLS].notna().any(axis=1).sum()
        if left:
            print(f"⚠️ 경고: 관측소 {e['관측소ID']} 센서 고장 구간 수온이 {left}행 남아 있음 → DB 전처리(전처리_보정.py) 확인 필요")
        else:
            print(f"확인: 관측소 {e['관측소ID']} 센서 고장 구간 수온 결측 처리됨 ({m.sum()}행)")


def add_features(df):
    """제안서의 '수온 변화량' = 이번 주 수온 − 같은 권역 직전 주 수온.
    직전 주의 기준은 v_학습데이터 뷰의 통합_1주전과 똑같이 맞춤 (같은 권역의 4~10일 전 보고가 있을 때만 계산, 없으면 결측).
    현재·과거 값만 쓰므로 데이터 누수 없음."""
    prev_date = df.groupby("권역명")[DATE_COL].shift()
    prev_temp = df.groupby("권역명")["수온_주평균"].shift()
    gap = (df[DATE_COL] - prev_date).dt.days
    df["수온_변화_1주"] = np.where(gap.between(4, 10), df["수온_주평균"] - prev_temp, np.nan)
    return df


def time_split(df):
    """시작일 순서로 7:1:2 분할. 같은 주(시작일)의 권역들은 같은 세트에 들어가게 날짜 단위로 자른다."""
    counts = df[DATE_COL].value_counts().sort_index()
    cum = df[DATE_COL].map(counts.cumsum()) / len(df)   # 그 날짜까지의 누적 비율
    tr, va, _ = SPLIT_RATIO
    return np.where(cum <= tr, "train", np.where(cum <= tr + va, "valid", "test"))


# ---------- 모델 정의 ----------
def logistic(pos_weight, C):
    # 결측 대체(중앙값)·표준화 기준은 fit 때 train으로만 계산되고, valid·test에는 그대로 적용됨
    return make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(C=C, class_weight="balanced", max_iter=2000),
    )


def lightgbm(pos_weight, **p):
    # 결측(NaN)은 그대로 넣음: 분기마다 결측을 어느 쪽으로 보낼지 학습함
    return LGBMClassifier(class_weight="balanced", subsample=0.8, subsample_freq=1,
                          colsample_bytree=0.8, random_state=SEED, verbose=-1, **p)


def xgboost(pos_weight, **p):
    # 결측(NaN)은 그대로 넣음. 클래스 불균형은 scale_pos_weight = 음성 수 / 양성 수 (train 기준)
    return XGBClassifier(scale_pos_weight=pos_weight, subsample=0.8, colsample_bytree=0.8,
                         eval_metric="logloss", random_state=SEED, n_jobs=-1, **p)


def grid(**kw):
    return [dict(zip(kw, v)) for v in itertools.product(*kw.values())]


# 이름: (사용 변수, 모델 생성 함수, 하이퍼파라미터 후보, 최종 선정 대상 여부)
MODELS = {
    "기준선":   (BASELINE_FEATURES, logistic, [{"C": 1.0}], False),
    "Logistic": (FEATURES, logistic, grid(C=[0.01, 0.03, 0.1, 0.3, 1, 3]), True),
    "LightGBM": (FEATURES, lightgbm, grid(n_estimators=[100, 300], learning_rate=[0.03, 0.1],
                                          num_leaves=[7, 15], min_child_samples=[10, 30]), True),
    "XGBoost":  (FEATURES, xgboost, grid(n_estimators=[100, 300], learning_rate=[0.03, 0.1],
                                         max_depth=[3, 5], min_child_weight=[1, 5]), True),
}


def scores(y, p):
    pred = (p >= THRESH).astype(int)
    return {"AUC": roc_auc_score(y, p), "PR_AUC": average_precision_score(y, p),
            "재현율": recall_score(y, pred), "정밀도": precision_score(y, pred, zero_division=0)}


# 데이터 준비 (전처리 + 분할)
df = add_features(load_data())
df["세트"] = time_split(df)
print(f"데이터 {len(df)}행 / 입력 변수 {len(FEATURES)}개")

확인: 관측소 22106 센서 고장 구간 수온 결측 처리됨 (2행)
데이터 2534행 / 입력 변수 11개


## 1. 비교 설정

In [4]:
ENV = ["수온_주평균", "수온_평년차", "수온_변화_1주", "기온_주평균", "풍속_주평균", "유의파고_주평균"]
SETS = {"환경 포함(11개)": FEATURES, "환경 제외(5개)": [c for c in FEATURES if c not in ENV]}
COMPARE_MODELS = ["Logistic", "LightGBM", "XGBoost"]
N_BOOT = 2000


def boot_diff(y, p_with, p_without, seed=0):
    """test 행을 복원추출해 (포함 − 제외) PR-AUC 차이의 분포를 구한다."""
    rng = np.random.default_rng(seed)
    y = np.asarray(y); diffs = []
    for _ in range(N_BOOT):
        i = rng.integers(0, len(y), len(y))
        if y[i].sum() == 0:
            continue
        diffs.append(average_precision_score(y[i], p_with[i]) - average_precision_score(y[i], p_without[i]))
    d = np.array(diffs)
    return np.percentile(d, 2.5), np.percentile(d, 97.5), (d > 0).mean()

## 2. 학습·평가

In [5]:
rows, boots = [], []
for k in HORIZONS:
    y_col = TARGET.format(k=k)
    d = df.dropna(subset=[y_col])
    tr, va, te = (d[d["세트"] == s] for s in ["train", "valid", "test"])
    ytr = tr[y_col].astype(int)
    pos_weight = (ytr == 0).sum() / max((ytr == 1).sum(), 1)
    for name in COMPARE_MODELS:
        _, build, params, _ = MODELS[name]
        test_p = {}
        for set_name, cols in SETS.items():
            best = max(params, key=lambda p: average_precision_score(
                va[y_col], build(pos_weight, **p).fit(tr[cols], ytr).predict_proba(va[cols])[:, 1]))
            model = build(pos_weight, **best).fit(tr[cols], ytr)
            pv = model.predict_proba(va[cols])[:, 1]
            pt = model.predict_proba(te[cols])[:, 1]
            test_p[set_name] = pt
            rows.append({"예측기간": k, "모델": name, "변수": set_name, "하이퍼파라미터": str(best),
                         "valid_PR_AUC": average_precision_score(va[y_col], pv),
                         "test_AUC": roc_auc_score(te[y_col], pt),
                         "test_PR_AUC": average_precision_score(te[y_col], pt),
                         "test_재현율": recall_score(te[y_col], pt >= THRESH)})
        lo, hi, share = boot_diff(te[y_col], *test_p.values())
        boots.append({"예측기간": k, "모델": name, "PR_AUC차이_95%하한": lo, "PR_AUC차이_95%상한": hi,
                      "포함이_더_높은_비율": share})
    print(f"{k}주 뒤 완료")

res = pd.DataFrame(rows).round(3)
bt = pd.DataFrame(boots).round(3)

1주 뒤 완료


2주 뒤 완료


4주 뒤 완료


## 3. 결과: valid·test PR-AUC (포함 / 제외)

In [6]:
res.pivot_table(index=["모델", "예측기간"], columns="변수",
               values=["valid_PR_AUC", "test_PR_AUC", "test_AUC"]).round(3)

test_AUC            test_PR_AUC            valid_PR_AUC           
변수            환경 제외(5개) 환경 포함(11개)   환경 제외(5개) 환경 포함(11개)    환경 제외(5개) 환경 포함(11개)
모델       예측기간                                                                    
LightGBM 1        0.820      0.782       0.318      0.258        0.545      0.540
         2        0.800      0.786       0.257      0.276        0.397      0.343
         4        0.762      0.693       0.296      0.214        0.539      0.590
Logistic 1        0.829      0.786       0.347      0.346        0.455      0.435
         2        0.785      0.763       0.268      0.306        0.516      0.519
         4        0.795      0.762       0.325      0.290        0.644      0.684
XGBoost  1        0.817      0.802       0.321      0.300        0.502      0.482
         2        0.803      0.773       0.270      0.291        0.385      0.348
         4        0.747      0.720       0.234      0.227        0.534      0.588

## 4. test PR-AUC 차이(포함 − 제외)의 부트스트랩 95% 구간
구간이 0을 포함하면 차이가 우연 범위 안이라는 뜻

In [7]:
bt

,예측기간,모델,PR_AUC차이_95%하한,PR_AUC차이_95%상한,포함이_더_높은_비율
0,1,Logistic,-0.076,0.080,0.465
1,1,LightGBM,-0.182,0.030,0.094
2,1,XGBoost,-0.125,0.055,0.245
3,2,Logistic,-0.043,0.101,0.771
4,2,LightGBM,-0.025,0.080,0.786
5,2,XGBoost,-0.049,0.090,0.632
6,4,Logistic,-0.186,0.103,0.292
7,4,LightGBM,-0.222,0.031,0.084
8,4,XGBoost,-0.100,0.085,0.411


## 5. 로지스틱이 고른 C

In [8]:
res[res["모델"] == "Logistic"][["예측기간", "변수", "하이퍼파라미터"]]

,예측기간,변수,하이퍼파라미터
0,1,환경 포함(11개),{'C': 0.01}
1,1,환경 제외(5개),{'C': 0.03}
6,2,환경 포함(11개),{'C': 0.3}
7,2,환경 제외(5개),{'C': 3}
12,4,환경 포함(11개),{'C': 0.01}
13,4,환경 제외(5개),{'C': 1}


## 6. 저장

In [9]:
out = res.merge(bt, on=["예측기간", "모델"])
out.to_csv(f"{OUT_DIR}/환경요인검증.csv", index=False, encoding="utf-8-sig")
print(f"저장: {OUT_DIR}/환경요인검증.csv")

저장: 결과/환경요인검증.csv


## 결론
- valid에서는 2·4주 뒤에 환경 변수를 넣은 모델이 높았고, test에서는 기간마다 엇갈림 (2주 뒤는 넣은 모델, 1·4주 뒤는 뺀 모델이 조금 높음)
- 세 모델 모두 차이의 95% 구간이 0을 포함 → 출현율·계절 정보 위에 환경 변수가 더하는 예측력은 이 자료에서 확인되지 않음
- 최종 모델은 valid 기준 절차대로 11개 변수 모델을 유지 (test 결과를 보고 변수를 바꾸지 않음)